# 实验日志与追踪：管理你的 ML 实验
> 难度标签：初级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：13_实用工具 | 源文件：实验日志与追踪.py | 核心功能：MLflow、实验记录、参数/指标追踪

## 概述

ML 实验管理需要记录每次实验的参数、指标、模型。MLflow 是最流行的开源工具。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import os
import json
import time
import logging
import numpy as np
# --- 导入库 ---
from datetime import datetime
from pathlib import Path

## 数学原理

### 1. 实验追踪的结构化表示

每个实验可表示为元组：

$$\text{Exp}_i = (\lambda_i, D_i, \theta_i, M_i, t_i)$$

- $\lambda_i$：超参数集合
- $D_i$：数据集标识
- $\theta_i$：模型参数
- $M_i$：评估指标集合
- $t_i$：时间戳

### 2. 指标的统计显著性

对比两个模型时，单次运行的结果不可靠。应使用多次运行的统计检验：

$$\bar{s} = \frac{1}{K}\sum_{k=1}^{K} s_k, \quad \text{SE} = \frac{\sigma_s}{\sqrt{K}}$$

95% 置信区间：$\bar{s} \pm 1.96 \cdot \text{SE}$

配对 t 检验比较两个模型：

$$t = \frac{\bar{s}_A - \bar{s}_B}{\text{SE}_{diff}}, \quad df = K - 1$$

### 3. JSON 格式的实验记录

结构化存储便于程序化分析：

$$\text{record} = \{\text{params}: \lambda, \text{metrics}: M, \text{timestamp}: t\}$$

支持后续的排序、过滤、聚合分析。

### 4. 超参数搜索的实验对比

| 实验 | 参数 $\lambda$ | 指标 $M$ |
|------|---------------|----------|
| Exp1 | $\{lr=0.01, d=3\}$ | $\{acc=0.85, f1=0.83\}$ |
| Exp2 | $\{lr=0.001, d=5\}$ | $\{acc=0.89, f1=0.87\}$ |

通过排序找到最优实验：$\lambda^* = \arg\max_i M_i(\text{acc})$

### 5. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `logging.info(f"acc={acc:.4f}")` | 记录指标到日志 |
| `json.dump(record, f)` | 序列化 $\text{Exp}_i \to \text{JSON}$ |
| `pd.DataFrame(records)` | 实验记录表，行=实验，列=指标 |
| `df.sort_values("accuracy", ascending=False)` | 按 $M_i$ 排序找最优 |
| `df.groupby("lr")["acc"].mean()` | 按参数分组统计平均指标 |

### 1. Python logging 模块基础

运行 1. Python logging 模块基础 的代码，观察算法在该环节的行为。

In [ ]:
print("=" * 60)
print("1. Python logging 模块基础")
print("=" * 60)

# 配置日志格式
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

logger = logging.getLogger('ml_experiment')

logger.info("实验开始")
logger.warning("学习率设置较高, 可能不稳定")
logger.error("这不是真正的错误, 只是演示")

# 日志级别
print(f"\n日志级别从低到高: DEBUG < INFO < WARNING < ERROR < CRITICAL")
print(f"设置 level=logging.INFO 时, DEBUG 不会输出")

### 2. 结构化实验日志

运行 2. 结构化实验日志 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("2. 结构化实验日志")
print("=" * 60)

class ExperimentLogger:
    """简单的实验日志记录器"""
    def __init__(self, experiment_name):
        self.name = experiment_name
        self.start_time = time.time()
# --- 赋值/配置 ---
        self.params = {}
        self.metrics = {}
        self.notes = []

        self.logger = logging.getLogger(f'ml.{experiment_name}')
        self.logger.info(f"=== 实验 [{experiment_name}] 开始 ===")

    def log_params(self, **kwargs):
        """记录超参数"""
        self.params.update(kwargs)
        param_str = ", ".join(f"{k}={v}" for k, v in kwargs.items())
        self.logger.info(f"参数: {param_str}")

    def log_metric(self, name, value, step=None):
        """记录指标"""
        if name not in self.metrics:
            self.metrics[name] = []
        self.metrics[name].append({'step': step, 'value': value})

    def log_note(self, note):
        """记录备注"""
        self.notes.append(note)
        self.logger.info(f"备注: {note}")

    def summary(self):
        """输出实验摘要"""
        elapsed = time.time() - self.start_time
        print(f"\n  实验 [{self.name}] 摘要:")
        print(f"  耗时: {elapsed:.2f}s")
# --- 输出结果 ---
        print(f"  参数: {self.params}")
        for name, values in self.metrics.items():
            final_val = values[-1]['value'] if values else None
            print(f"  {name}: {final_val}")
        if self.notes:
# --- 输出结果 ---
            print(f"  备注: {self.notes}")

# 使用
exp = ExperimentLogger("iris_rf_baseline")
exp.log_params(n_estimators=100, max_depth=None, random_state=42)

from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

iris = load_iris()
clf = RandomForestClassifier(n_estimators=100, random_state=42)
scores = cross_val_score(clf, iris.data, iris.target, cv=5)

exp.log_metric('accuracy_mean', scores.mean())
exp.log_metric('accuracy_std', scores.std())
exp.log_note("baseline模型, 未调参")
exp.summary()

### 3. JSON 实验记录

运行 3. JSON 实验记录 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("3. JSON 实验记录")
print("=" * 60)

class ExperimentTracker:
    """基于JSON的实验追踪器"""
    def __init__(self, log_dir='experiments'):
        self.log_dir = Path(log_dir)
        self.log_dir.mkdir(exist_ok=True)
# --- 赋值/配置 ---
        self.current_experiment = None

    def start_experiment(self, name, params=None):
        """开始新实验"""
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        exp_id = f"{name}_{timestamp}"
        self.current_experiment = {
# --- 继续 ---
            'id': exp_id,
            'name': name,
            'timestamp': timestamp,
            'params': params or {},
            'metrics': {},
# --- 继续 ---
            'status': 'running'
        }
        print(f"  开始实验: {exp_id}")
        return exp_id

    def log_metrics(self, **metrics):
        """记录指标"""
        if self.current_experiment:
            for k, v in metrics.items():
                self.current_experiment['metrics'][k] = v

    def finish_experiment(self, status='completed'):
        """结束并保存实验"""
        if self.current_experiment:
            self.current_experiment['status'] = status
            self.current_experiment['duration'] = time.time()

            filepath = self.log_dir / f"{self.current_experiment['id']}.json"
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(self.current_experiment, f, indent=2, ensure_ascii=False)
            print(f"  实验已保存: {filepath}")
            self.current_experiment = None

    def load_experiment(self, filepath):
        """加载实验记录"""
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)

    def compare_experiments(self, exp_files):
        """对比多个实验"""
        experiments = []
        for f in exp_files:
            data = self.load_experiment(f)
# --- 继续 ---
            experiments.append(data)

        print(f"\n  实验对比 ({len(experiments)}个):")
        print(f"  {'实验名':<25} {'状态':<10} {'指标'}")
        print(f"  {'-'*25} {'-'*10} {'-'*30}")
        for exp in experiments:
            metrics_str = ", ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}"
# --- 循环处理 ---
                                   for k, v in exp.get('metrics', {}).items())
            print(f"  {exp['name']:<25} {exp['status']:<10} {metrics_str}")

# 使用
tracker = ExperimentTracker('my_experiments')

# 实验1
tracker.start_experiment('rf_baseline', {'n_estimators': 100})
tracker.log_metrics(accuracy=0.95, f1=0.94)
tracker.finish_experiment()

# 实验2
tracker.start_experiment('rf_tuned', {'n_estimators': 200, 'max_depth': 5})
tracker.log_metrics(accuracy=0.97, f1=0.96)
tracker.finish_experiment()

# 对比
import glob
exp_files = sorted(glob.glob('my_experiments/*.json'))
if exp_files:
    tracker.compare_experiments(exp_files)

### 4. 训练过程追踪

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
print("\n" + "=" * 60)
print("4. 训练过程追踪")
print("=" * 60)

# 模拟训练过程的日志
np.random.seed(42)

class TrainingLogger:
    """记录训练过程中的损失和指标变化"""
    def __init__(self):
        self.history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'val_acc': []}

    def log_epoch(self, epoch, train_loss, val_loss, val_acc):
        self.history['epoch'].append(epoch)
        self.history['train_loss'].append(train_loss)
        self.history['val_loss'].append(val_loss)
        self.history['val_acc'].append(val_acc)

    def print_progress(self, epoch, total_epochs):
        """打印训练进度"""
        bar_width = 30
        progress = (epoch + 1) / total_epochs
        filled = int(bar_width * progress)
# --- 继续 ---
        bar = '#' * filled + '-' * (bar_width - filled)

        train_loss = self.history['train_loss'][-1]
        val_loss = self.history['val_loss'][-1]
        val_acc = self.history['val_acc'][-1]

        print(f"  Epoch {epoch+1:>3}/{total_epochs} |{bar}| "
              f"loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    def best_epoch(self):
        """找到最佳epoch"""
        best_idx = np.argmin(self.history['val_loss'])
        return {
            'epoch': self.history['epoch'][best_idx],
# --- 继续 ---
            'train_loss': self.history['train_loss'][best_idx],
            'val_loss': self.history['val_loss'][best_idx],
            'val_acc': self.history['val_acc'][best_idx]
        }

# 模拟训练
train_log = TrainingLogger()
base_loss = 2.0
for epoch in range(20):
    train_loss = base_loss * np.exp(-0.15 * epoch) + np.random.normal(0, 0.02)
    val_loss = base_loss * np.exp(-0.12 * epoch) + np.random.normal(0, 0.03)
# --- 数值计算 ---
    val_acc = min(0.5 + 0.025 * epoch + np.random.normal(0, 0.01), 0.99)
    train_log.log_epoch(epoch, train_loss, val_loss, val_acc)
    if (epoch + 1) % 5 == 0:
        train_log.print_progress(epoch, 20)

best = train_log.best_epoch()
print(f"\n  最佳epoch: {best['epoch']+1}, val_loss={best['val_loss']:.4f}, val_acc={best['val_acc']:.4f}")

### 5. 实验管理最佳实践

运行 5. 实验管理最佳实践 的代码，观察算法在该环节的行为。

In [ ]:
print("\n" + "=" * 60)
print("5. 实验管理最佳实践")
print("=" * 60)

print("""
实验记录应包含:
  1. 超参数: 所有影响结果的配置
  2. 数据信息: 数据集名称、大小、划分方式
  3. 环境信息: Python版本、库版本、硬件(CPU/GPU)
# --- 继续 ---
  4. 指标: 训练/验证/测试集上的各项指标
  5. 时间: 开始时间、结束时间、耗时
  6. 随机种子: 保证可复现
  7. 代码版本: git commit hash (如有)

工具推荐 (从轻到重):
  1. JSON/CSV手动记录 → 适合小项目、学习
  2. TensorBoard → PyTorch/TF官方支持, 可视化好
  3. MLflow → 开源, 功能全面, 支持模型管理
  4. Weights & Biases → 云服务, 协作方便

命名规范:
  实验名 = 模型_数据集_关键参数
  例如: rf_iris_n100 / bert_sst2_lr2e-5
""")

# 清理
import shutil
if os.path.exists('my_experiments'):
    shutil.rmtree('my_experiments')
    print("已清理临时实验目录")

## 关键代码解释

```python
import mlflow
mlflow.log_param("learning_rate", 0.01)
mlflow.log_metric("accuracy", 0.95)
mlflow.sklearn.log_model(model, "model")
```

## 注意事项

1. 从第一个实验就开始记录
2. 记录足够信息以便复现
3. 定期清理不好的实验

## 总结与延伸

以上代码展示了 **实验日志与追踪** 的完整流程。

**解读要点**：
- 关注工具的 **输入输出格式** 是否正确
- 观察数据加载和预处理的效率
- 检查模型保存/加载后的一致性

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Weights & Biases**：更现代的实验追踪
- **DVC**：数据版本控制
- **实验复现性**：随机种子 + 环境锁定
